# Free real AI video generator for CreatorAI Studio

Runs Alibaba's open-source **Wan2.1 (1.3B)** text-to-video model on Colab's free T4 GPU — real AI-generated video, not a pan/zoom effect, with no payment ever required.

**How to use:**
1. `Runtime` menu → `Change runtime type` → select **T4 GPU** → Save.
2. `Runtime` menu → `Run all`.
3. Wait for the last cell to print a public URL ending in `.gradio.live` (first run takes ~5-10 min to download model weights; later runs are much faster).
4. Copy that URL into `.env.local` in the project as `COLAB_VIDEO_ENDPOINT=https://xxxxx.gradio.live`, then restart the dev server.
5. Select "Wan 2.1 (Colab)" as the provider in AI Video Generation.

**Important:** this URL is only valid while this Colab tab stays open and connected. Free Colab sessions disconnect after ~90 minutes idle or ~12 hours total. When it disconnects, re-run this notebook and update the URL again.

In [ ]:
!pip install -q diffusers transformers accelerate imageio imageio-ffmpeg gradio ftfy

In [ ]:
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video

assert torch.cuda.is_available(), "No GPU detected — go to Runtime > Change runtime type > T4 GPU, then rerun."

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.bfloat16)
pipe.to("cuda")

NEGATIVE_PROMPT = (
    "Bright tones, overexposed, static, blurred details, subtitles, style, works, "
    "paintings, images, static, overall gray, worst quality, low quality, JPEG "
    "compression residue, ugly, incomplete, extra fingers, poorly drawn hands, "
    "poorly drawn faces, deformed, disfigured, malformed limbs, fused fingers, "
    "still picture, cluttered background, three legs, many people in the background, "
    "walking backwards"
)
print("Model loaded.")

In [ ]:
import gradio as gr

def generate(prompt, duration_seconds, seed):
    fps = 16
    # Wan requires num_frames = 4k+1; clamp duration to a sane 1-6s range for T4 generation time
    duration_seconds = max(1, min(6, float(duration_seconds)))
    num_frames = int(round(duration_seconds * fps / 4)) * 4 + 1

    generator = torch.Generator(device="cuda").manual_seed(int(seed))
    output = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        height=480,
        width=832,
        num_frames=num_frames,
        guidance_scale=5.0,
        num_inference_steps=30,
        generator=generator,
    ).frames[0]

    out_path = f"/content/out_{int(seed)}.mp4"
    export_to_video(output, out_path, fps=fps)
    return out_path

demo = gr.Interface(
    fn=generate,
    inputs=[
        gr.Textbox(label="Prompt", placeholder="A golden retriever puppy playing in a sunny park"),
        gr.Slider(1, 6, value=4, step=1, label="Duration (seconds)"),
        gr.Number(value=42, label="Seed"),
    ],
    outputs=gr.Video(label="Generated video"),
    title="Wan2.1 (1.3B) — free text-to-video",
)
demo.launch(share=True)